In [1]:
import pandas as pd 
import numpy as np
import requests

In [2]:
raw = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/raw_movies_data.csv')
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/main.csv')

In [3]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4803 non-null   int64  
 1   genres                4803 non-null   object 
 2   homepage              1712 non-null   object 
 3   id                    4803 non-null   int64  
 4   keywords              4803 non-null   object 
 5   original_language     4803 non-null   object 
 6   original_title        4803 non-null   object 
 7   overview              4800 non-null   object 
 8   popularity            4803 non-null   float64
 9   production_companies  4803 non-null   object 
 10  production_countries  4803 non-null   object 
 11  release_date          4802 non-null   object 
 12  revenue               4803 non-null   int64  
 13  runtime               4801 non-null   float64
 14  spoken_languages      4803 non-null   object 
 15  status               

In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   popularity         4803 non-null   object
 3   runtime            4803 non-null   object
 4   tagline            4803 non-null   object
 5   title              4803 non-null   object
 6   vote_average       4803 non-null   object
 7   vote_count         4803 non-null   object
 8   genre_names        4775 non-null   object
 9   keywords_names     4391 non-null   object
 10  movie_text         4803 non-null   object
dtypes: object(11)
memory usage: 412.9+ KB


In [5]:
df.drop(columns=['popularity', 'vote_average', 'vote_count', 'keywords_names'], inplace=True)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 7 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   runtime            4803 non-null   object
 3   tagline            4803 non-null   object
 4   title              4803 non-null   object
 5   genre_names        4775 non-null   object
 6   movie_text         4803 non-null   object
dtypes: object(7)
memory usage: 262.8+ KB


In [7]:
df['id'] = raw['id']


In [8]:
df.head(20)

,original_language,overview,runtime,tagline,title,genre_names,movie_text,id
0,en,"In the 22nd century, a paraplegic Marine is di...",2.46808510638298,Enter the World of Pandora.,Avatar,"Action, Adventure, Fantasy, Science Fiction","Action, Adventure, Fantasy, Science Fiction cu...",19995
1,en,"Captain Barbossa, long believed to be dead, ha...",2.76595744680851,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,"Adventure, Fantasy, Action","Adventure, Fantasy, Action ocean, drug abuse, ...",285
2,en,A cryptic message from Bond’s past sends him o...,1.87234042553192,A Plan No One Escapes,Spectre,"Action, Adventure, Crime","Action, Adventure, Crime spy, based on novel, ...",206647
3,en,Following the death of District Attorney Harve...,2.59574468085106,The Legend Ends,The Dark Knight Rises,"Action, Crime, Drama, Thriller","Action, Crime, Drama, Thriller dc comics, crim...",49026
4,en,"John Carter is a war-weary, former military ca...",1.19148936170213,"Lost in our world, found in another.",John Carter,"Action, Adventure, Science Fiction","Action, Adventure, Science Fiction based on no...",49529
5,en,The seemingly invincible Spider-Man goes up ag...,1.48936170212766,The battle within.,Spider-Man 3,"Fantasy, Action, Adventure","Fantasy, Action, Adventure dual identity, amne...",559
6,en,When the kingdom's most wanted-and most charmi...,-0.170212765957447,They're taking adventure to new lengths.,Tangled,"Animation, Family","Animation, Family hostage, magic, horse, fairy...",38757
7,en,When Tony Stark tries to jumpstart a dormant p...,1.57446808510638,A New Age Has Come.,Avengers: Age of Ultron,"Action, Adventure, Science Fiction","Action, Adventure, Science Fiction marvel comi...",99861
8,en,"As Harry begins his sixth year at Hogwarts, he...",2.08510638297872,Dark Secrets Revealed,Harry Potter and the Half-Blood Prince,"Adventure, Fantasy, Family","Adventure, Fantasy, Family witch, magic, broom...",767
9,en,Fearing the actions of a god-like Super Hero l...,2,Justice or revenge,Batman v Superman: Dawn of Justice,"Action, Adventure, Fantasy","Action, Adventure, Fantasy dc comics, vigilant...",209112


In [9]:
df['genre_names'] = df['genre_names'].fillna(' ')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   runtime            4803 non-null   object
 3   tagline            4803 non-null   object
 4   title              4803 non-null   object
 5   genre_names        4803 non-null   object
 6   movie_text         4803 non-null   object
 7   id                 4803 non-null   int64 
dtypes: int64(1), object(7)
memory usage: 300.3+ KB


In [10]:
df['id'].head()

0     19995
1       285
2    206647
3     49026
4     49529
Name: id, dtype: int64

In [11]:
API_KEY = '28ed9fa013cb76618abf444748fd601d'

In [12]:
import pandas as pd
import requests
import time
from concurrent.futures import ThreadPoolExecutor, as_completed



BASE_URL = "https://api.themoviedb.org/3/movie/"
IMG_BASE = "https://image.tmdb.org/t/p/original"

def fetch_poster(movie_id):
    """
    Ultra-fast thread-safe reporter.
    Handles small delays to avoid TMDB rate limits.
    """
    url = f"{BASE_URL}{movie_id}?api_key={API_KEY}"

    try:
        r = requests.get(url, timeout=5)

        # API Limit Protection
        time.sleep(0.05)

        if r.status_code != 200:
            return None

        data = r.json()
        poster_path = data.get("poster_path")

        if poster_path:
            return IMG_BASE + poster_path
        return None

    except:
        return None


# ---------------------------------------------------
# APPLY TO FULL DATASET WITH 16 HIGH-SPEED WORKERS
# ---------------------------------------------------

movie_ids = df["id"].tolist()
results = {}

print("⚡ Starting ultra-fast 16-thread poster fetch...")

with ThreadPoolExecutor(max_workers=16) as executor:
    futures = {executor.submit(fetch_poster, mid): mid for mid in movie_ids}

    for future in as_completed(futures):
        mid = futures[future]
        try:
            results[mid] = future.result()
        except:
            results[mid] = None


# Map results back to DataFrame
df["poster_url"] = df["id"].map(results)

# Save Output
df.to_csv("movies_with_posters.csv", index=False)

print("🔥 DONE — Ultra-fast poster fetch complete (16 workers)")


⚡ Starting ultra-fast 16-thread poster fetch...
🔥 DONE — Ultra-fast poster fetch complete (16 workers)


In [13]:
df2 = pd.read_csv('/media/prince/5A4E832F4E83034D/Movie recomender/zDeployment/movies_with_posters.csv')

In [15]:
df2['poster_url'] = df2['poster_url'].fillna('')
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 9 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   original_language  4803 non-null   object
 1   overview           4803 non-null   object
 2   runtime            4803 non-null   object
 3   tagline            4803 non-null   object
 4   title              4803 non-null   object
 5   genre_names        4803 non-null   object
 6   movie_text         4803 non-null   object
 7   id                 4803 non-null   int64 
 8   poster_url         4803 non-null   object
dtypes: int64(1), object(8)
memory usage: 337.8+ KB


In [16]:
df2.to_csv('final_list.csv', index=False)